# 🫧 K-Means Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> K-means groups customers into segments by finding cluster centers that minimize within-cluster distance. The result: actionable customer segments you can market to differently.

### Dataset
**E-commerce Customer Segmentation via RFM Analysis**
RFM (Recency, Frequency, Monetary) is the gold standard framework for customer analytics. We build it from a UK online retail transaction dataset and cluster customers into distinct behavioral segments.

### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
print("\u2713 Setup complete")

In [ ]:
# UK Online Retail dataset — build RFM features
# Source: UCI ML Repository — real e-commerce transactions 2010-2011
import numpy as np
import pandas as pd

try:
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
    retail = pd.read_excel(url, engine='openpyxl')
    retail = retail.dropna(subset=['CustomerID'])
    retail = retail[retail['Quantity'] > 0]
    retail['TotalPrice'] = retail['Quantity'] * retail['UnitPrice']
    snapshot_date = retail['InvoiceDate'].max()
    rfm = retail.groupby('CustomerID').agg(
        Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('TotalPrice', 'sum')
    ).reset_index()
    print("Loaded UK Online Retail dataset from UCI")
except:
    # Synthetic RFM — realistic e-commerce customer distribution
    np.random.seed(42)
    n = 4372  # matches real dataset customer count
    # Champions: low recency, high freq, high spend
    n_champ = 500
    # At-Risk: high recency, was good
    n_risk = 800
    # Hibernating: very high recency, low freq
    n_hiber = 1200
    # New: low recency, low freq
    n_new = 600
    # Loyal: medium recency, high freq
    n_loyal = 700
    # Others
    n_other = n - n_champ - n_risk - n_hiber - n_new - n_loyal

    rfm_data = []
    for seg, nr, rec_mu, rec_sd, freq_mu, freq_sd, mon_mu, mon_sd in [
        ('Champions',   n_champ, 15,  10, 12,  4,  1500, 600),
        ('At-Risk',     n_risk,  120, 40, 8,   3,  900,  400),
        ('Hibernating', n_hiber, 280, 60, 2,   1,  200,  100),
        ('New',         n_new,   20,  12, 1,   0.5,150,   80),
        ('Loyal',       n_loyal, 45,  20, 10,  3,  1100, 450),
        ('Others',      n_other, 180, 70, 3,   2,  350,  200),
    ]:
        rfm_data.append(pd.DataFrame({
            'CustomerID': range(len(rfm_data)*1000, len(rfm_data)*1000+nr),
            'Recency':    np.abs(np.random.normal(rec_mu, rec_sd, nr)).astype(int).clip(1,365),
            'Frequency':  np.abs(np.random.normal(freq_mu,freq_sd,nr)).astype(int).clip(1,50),
            'Monetary':   np.abs(np.random.normal(mon_mu, mon_sd, nr)).clip(10, 10000).round(2)
        }))
    rfm = pd.concat(rfm_data, ignore_index=True)
    print("Using synthetic RFM dataset (realistic UK retail properties)")

print(f"\nRFM dataset: {rfm.shape[0]:,} customers")
print(f"\nRFM Summary:")
print(rfm[['Recency','Frequency','Monetary']].describe().round(1).to_string())
print("\nInterpretation:")
print("  Recency:   days since last purchase (lower = better)")
print("  Frequency: number of distinct orders (higher = better)")
print("  Monetary:  total spend in GBP (higher = better)")

## 📐 Part 1 — K-Means Algorithm

At each iteration:
1. **Assign** each point to its nearest centroid (minimize Euclidean distance)
2. **Update** each centroid to the mean of its assigned points
3. Repeat until assignments stop changing

**Key property:** K-means minimizes **Within-Cluster Sum of Squares (WCSS)**.
Always standardize first — monetary values in £ would completely dominate recency in days.

In [ ]:
# Standardize RFM (critical — Monetary in £ vs Recency in days otherwise)
X_rfm = rfm[['Recency','Frequency','Monetary']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# Note: we invert Recency so higher always means "better" for interpretation
X_scaled[:,0] *= -1  # higher score = more recent

# Elbow method — find optimal K
wcss = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), wcss, 'o-', color='#1e5fa8', lw=2, markersize=7)
axes[0].set_xlabel('Number of Segments (K)')
axes[0].set_ylabel('WCSS (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method\n(look for the "elbow" — diminishing returns)')

axes[1].plot(list(K_range), sil_scores, 's-', color='#e85d20', lw=2, markersize=7)
best_k = list(K_range)[sil_scores.index(max(sil_scores))]
axes[1].axvline(best_k, color='#1a7a45', lw=1.5, ls='--',
                label=f'Best K={best_k} (silhouette={max(sil_scores):.3f})')
axes[1].set_xlabel('Number of Segments (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — higher = more distinct segments')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"\n\u2714 Silhouette suggests K={best_k} customer segments")

In [ ]:
# Fit final model and name the segments
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm['Segment'] = km_final.fit_predict(X_scaled)

# Profile each segment on original scale
seg_profile = rfm.groupby('Segment').agg(
    N=('CustomerID','count'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean')
).round(1)
seg_profile['% of customers'] = (seg_profile['N'] / len(rfm) * 100).round(1)

# Name segments based on profile
def name_segment(row):
    if row['Avg_Recency'] < 40 and row['Avg_Frequency'] > 7:
        return 'Champions'
    elif row['Avg_Recency'] < 60 and row['Avg_Frequency'] > 4:
        return 'Loyal Customers'
    elif row['Avg_Recency'] > 150:
        return 'Lost / Hibernating'
    elif row['Avg_Frequency'] <= 2 and row['Avg_Recency'] < 50:
        return 'New Customers'
    else:
        return 'At-Risk'

seg_profile['Segment Name'] = seg_profile.apply(name_segment, axis=1)
print("=== Customer Segment Profiles ===\n")
print(seg_profile[['Segment Name','N','% of customers',
                    'Avg_Recency','Avg_Frequency','Avg_Monetary']].to_string())

In [ ]:
# Visualize segments in 2D (PCA)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

seg_names = rfm['Segment'].map(
    seg_profile['Segment Name'].to_dict())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_seg = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1','#0e7490']

for seg_id in rfm['Segment'].unique():
    mask = rfm['Segment'] == seg_id
    name = seg_profile.loc[seg_id,'Segment Name']
    axes[0].scatter(X_pca[mask,0], X_pca[mask,1], s=12, alpha=0.5,
                   color=colors_seg[seg_id % len(colors_seg)], label=name)

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% variance)')
axes[0].set_title('Customer Segments in PCA Space')
axes[0].legend(fontsize=9)

# Radar/bar chart of segment profiles
seg_profile_norm = seg_profile[['Avg_Recency','Avg_Frequency','Avg_Monetary']].copy()
for col in seg_profile_norm.columns:
    seg_profile_norm[col] = (seg_profile_norm[col] - seg_profile_norm[col].min()) /                              (seg_profile_norm[col].max() - seg_profile_norm[col].min())
seg_profile_norm.index = seg_profile['Segment Name']

seg_profile_norm.plot(kind='bar', ax=axes[1], color=['#1e5fa8','#e85d20','#1a7a45'],
                      edgecolor='white', width=0.7)
axes[1].set_title('Normalized Segment Profiles\n(higher = better for Frequency/Monetary, lower = better for Recency)')
axes[1].set_xlabel('Segment')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()
print("\n\u2714 Business actions by segment:")
print("  Champions:        Reward loyalty — offer VIP program")
print("  Loyal Customers:  Upsell — they're engaged and spending")
print("  At-Risk:          Re-engagement campaign — discount or win-back email")
print("  New Customers:    Onboarding nurture — encourage 2nd purchase")
print("  Lost/Hibernating: Low-cost reactivation attempt or write off")

In [ ]:
answers = {
    "q1": "",  # What does K-means minimize, and why must features be standardized first?
    "q2": "",  # What does the silhouette score measure? What does a value near 0 mean?
    "q3": "",  # What is the elbow method and why doesn't it always give a clear answer?
    "q4": "",  # Why is K-means sensitive to initialization, and how does n_init help?
    "q5": "",  # For each customer segment identified, describe ONE concrete marketing action.
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "K-Means Clustering"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*